# Agent 11 â€” Portfolio Construction

**What this notebook does:**  
Combines the financial metrics (Sharpe ratio) and ESG scores to rank all 50+ companies, apply exclusion rules, and select the final 15â€“25 holdings with weights.

**How to present this to investors:**  
> *Our portfolio construction agent uses a transparent composite score combining financial quality and ESG quality. We apply explicit exclusion rules, enforce diversification constraints (max 10% per stock, min 5 sectors), and the final weights are rule-based â€” not a black box.*

In [1]:
import pandas as pd
import numpy as np
from datetime import date
import glob
import os

esg_files = sorted(glob.glob('../outputs/scores/esg_scores_*.csv'))
fin_files = sorted(glob.glob('../outputs/scores/financial_metrics_*.csv'))
bio_files = sorted(glob.glob('../outputs/scores/biodiversity_scores_*.csv'))
eu_files  = sorted(glob.glob('../outputs/scores/eu_regulation_*.csv'))
gw_files  = sorted(glob.glob('../outputs/scores/greenwashing_scores_*.csv'))
master_files = sorted(glob.glob('../outputs/scores/master_dataset_*.csv'))

if not esg_files or not fin_files:
    raise FileNotFoundError('Run Agent 10 and Agent 5/6 first.')

df_esg = pd.read_csv(esg_files[-1])
df_fin = pd.read_csv(fin_files[-1])
df_bio = pd.read_csv(bio_files[-1]) if bio_files else None
df_eu  = pd.read_csv(eu_files[-1])  if eu_files  else None
df_gw  = pd.read_csv(gw_files[-1])  if gw_files  else None

# Load ticker bridge: Bloomberg ticker (bb_ticker) ↔ Yahoo Finance ticker (yf_ticker)
# ESG scores use Bloomberg tickers; financial metrics use Yahoo Finance tickers.
# The master dataset contains both — use it as the bridge.
if master_files:
    master = pd.read_csv(master_files[-1], usecols=["ticker", "yf_ticker"])
    master = master.dropna(subset=["yf_ticker"]).drop_duplicates(subset=["ticker"])
    ticker_bridge = master.rename(columns={"ticker": "bb_ticker"})
    print(f"Ticker bridge loaded: {len(ticker_bridge)} mappings")
else:
    ticker_bridge = None
    print("WARNING: No master dataset found — ticker bridge unavailable")

print(f'ESG scores:          {len(df_esg)} companies  (Bloomberg tickers)')
print(f'Financial metrics:   {len(df_fin)} companies  (Yahoo Finance tickers)')
print(f'Biodiversity scores: {len(df_bio) if df_bio is not None else 0} companies')
print(f'EU regulation:       {len(df_eu)  if df_eu  is not None else 0} companies')
print(f'Greenwashing scores: {len(df_gw)  if df_gw  is not None else 0} companies')

Ticker bridge loaded: 167 mappings
ESG scores:          167 companies  (Bloomberg tickers)
Financial metrics:   164 companies  (Yahoo Finance tickers)
Biodiversity scores: 167 companies
EU regulation:       167 companies
Greenwashing scores: 0 companies


## Step 1 â€” Merge ESG and financial data

We combine both datasets on the ticker symbol.

In [2]:
TICKER_COL = 'ticker'   # Bloomberg ticker — primary key throughout

# Step 1a: Add yf_ticker to ESG dataframe via bridge
if ticker_bridge is not None:
    df_esg = df_esg.merge(ticker_bridge, left_on=TICKER_COL, right_on="bb_ticker", how="left")
    df_esg = df_esg.drop(columns=["bb_ticker"], errors="ignore")
    n_matched = df_esg["yf_ticker"].notna().sum()
    print(f"Ticker bridge: {n_matched}/{len(df_esg)} ESG companies matched to Yahoo Finance tickers")

# Step 1b: Add company name from master to ESG (useful for display)
master_full = pd.read_csv(master_files[-1], usecols=["ticker", "idBbGlobalCompanyName"]).drop_duplicates(subset=["ticker"])
df_esg = df_esg.merge(master_full, on="ticker", how="left")

# Step 1c: Merge ESG + Financial via yf_ticker
if "yf_ticker" in df_esg.columns:
    combined = df_esg.merge(df_fin, left_on="yf_ticker", right_on=TICKER_COL,
                            how="inner", suffixes=("", "_fin"))
    # Restore Bloomberg ticker as the primary key
    if TICKER_COL + "_fin" in combined.columns:
        combined = combined.drop(columns=[TICKER_COL + "_fin"])
    print(f"After ESG + Financial merge (via yf_ticker): {len(combined)} companies")
else:
    # Fallback: direct ticker merge (may miss many)
    combined = df_esg.merge(df_fin, on=TICKER_COL, how="inner", suffixes=("", "_fin"))
    print(f"After ESG + Financial merge (direct ticker): {len(combined)} companies")

if len(combined) == 0:
    raise ValueError("Merge produced 0 rows — check ticker bridge")

# Step 1d: Merge biodiversity
if df_bio is not None:
    bio_keep = [TICKER_COL] + [c for c in df_bio.columns
                if c in ["biodiversity_score", "nature_risk_tier", "encore_score", "aqueduct_score"]]
    combined = combined.merge(df_bio[bio_keep], on=TICKER_COL, how="left")
    print(f"Biodiversity merged: {combined['biodiversity_score'].notna().sum()} companies")
else:
    combined["biodiversity_score"] = None
    combined["nature_risk_tier"]   = None
    print("No biodiversity file found.")

# Step 1e: Merge EU regulation
if df_eu is not None:
    eu_keep = [TICKER_COL] + [c for c in df_eu.columns
               if any(k in c.lower() for k in ["eligib", "align", "dnsh", "sfdr", "taxonomy"])]
    eu_keep = list(dict.fromkeys(eu_keep))
    combined = combined.merge(df_eu[eu_keep], on=TICKER_COL, how="left")
    print(f"EU regulation merged: {len(eu_keep)-1} columns added")
else:
    print("No EU regulation file found.")

# Step 1f: Merge greenwashing
if df_gw is not None:
    gw_keep = [TICKER_COL] + [c for c in df_gw.columns
               if any(k in c.lower() for k in ["gw_exclude", "gw_watchlist", "gw_score", "gw_high"])]
    gw_keep = list(dict.fromkeys(gw_keep))
    combined = combined.merge(df_gw[gw_keep], on=TICKER_COL, how="left")
    print(f"Greenwashing merged: {combined['gw_exclude'].notna().sum()} companies assessed")
else:
    combined["gw_exclude"]   = False
    combined["gw_watchlist"] = False
    print("No greenwashing file found.")

print(f"\nFinal combined: {len(combined)} companies, {len(combined.columns)} columns")
combined[[TICKER_COL, "idBbGlobalCompanyName", "ESG_score", "sharpe_ratio"]].head(5)

Ticker bridge: 167/167 ESG companies matched to Yahoo Finance tickers
After ESG + Financial merge (via yf_ticker): 164 companies
Biodiversity merged: 164 companies
EU regulation merged: 6 columns added
No greenwashing file found.

Final combined: 164 companies, 41 columns


,ticker,idBbGlobalCompanyName,ESG_score,sharpe_ratio
0,SEBA,Skandinaviska Enskilda Banken AB,48.33,0.550
1,ALV,Allianz SE,66.37,0.603
2,E7V,Rexel SA,54.64,0.471
3,IES,Intesa Sanpaolo SpA,59.19,0.557
4,RWE,RWE AG,47.23,0.674


## Step 2 â€” Apply exclusions

Before scoring, we remove companies that fail our minimum standards.  
This is the greenwashing / controversy watchlist filter.

**Adjust the thresholds below to match your mandate.**

In [3]:
exclusions = []

# ── Rule 0: Parent/subsidiary deduplication ────────────────────────────────
PARENT_SUB_PAIRS = [
    ("CRIN", "ZS3"),   # UniCredit SpA owns ~52% of FinecoBank Banca Fineco
    ("INN1", "6GF"),   # ING Groep NV owns ~75% of ING Bank Slaski SA
    ("ASG",  "B7A"),   # Assicurazioni Generali owns ~50% of Banca Generali
    ("KDB",  "KB9"),   # KBC Ancora is a holding vehicle for KBC Group shares
]
eligible_set = set(combined[TICKER_COL].tolist())
for parent, sub in PARENT_SUB_PAIRS:
    if parent in eligible_set and sub in eligible_set:
        exclusions.append((sub, f"Subsidiary of {parent} — excluded to prevent carbon double-counting"))
        print(f"Parent/sub pair: keeping {parent}, excluding {sub}")

# ── Rule 0b: LOW_DATA — no ESG pillar data ────────────────────────────────
if "esg_data_flag" in combined.columns:
    low_data = combined[combined["esg_data_flag"] == "LOW_DATA"][TICKER_COL].tolist()
    for t in low_data:
        exclusions.append((t, "No ESG data (all three pillars NaN) — excluded per hallucination control policy"))
    if low_data:
        print(f"LOW_DATA exclusions: {low_data}")

# ── Rule 1: Greenwashing — Agent 9 automatic exclusion ──────────────────────
if "gw_exclude" in combined.columns:
    for _, row in combined[combined["gw_exclude"] == True].iterrows():
        exclusions.append((row[TICKER_COL], "Greenwashing: HIGH on 3+ dimensions (Agent 9)"))

# ── Rule 2: Very High nature risk — Agent 7 ──────────────────────────────────
if "nature_risk_tier" in combined.columns:
    for _, row in combined[combined["nature_risk_tier"] == "Very High"].iterrows():
        exclusions.append((row[TICKER_COL], "Biodiversity: Very High nature risk tier (Agent 7)"))

# ── Rule 3: ESG floor — bottom 10% ───────────────────────────────────────────
ESG_FLOOR = combined["ESG_score"].quantile(0.10)
for _, row in combined[combined["ESG_score"] < ESG_FLOOR].iterrows():
    esg_val = row['ESG_score']
    exclusions.append((row[TICKER_COL], f"ESG score below floor ({esg_val:.1f} < {ESG_FLOOR:.1f})"))

# ── Rule 4: Negative Sharpe ratio ─────────────────────────────────────────────
for _, row in combined[combined["sharpe_ratio"] < 0].iterrows():
    sr_val = row['sharpe_ratio']
    exclusions.append((row[TICKER_COL], f"Negative Sharpe ratio ({sr_val:.2f})"))

# ── Rule 5: Manual overrides ──────────────────────────────────────────────────
MANUAL_EXCLUSIONS = []
exclusions.extend(MANUAL_EXCLUSIONS)

excluded_tickers = list(set([t for t, _ in exclusions]))
eligible = combined[~combined[TICKER_COL].isin(excluded_tickers)].copy()

print(f"\nCompanies excluded: {len(excluded_tickers)}")
print(f"Eligible for portfolio: {len(eligible)}")
import os; os.makedirs("../outputs/portfolio", exist_ok=True)
excl_df = pd.DataFrame(exclusions, columns=["ticker", "reason"]).drop_duplicates()
excl_df.to_csv("../outputs/portfolio/exclusions.csv", index=False)
print("Exclusion log saved.")
excl_df

Parent/sub pair: keeping CRIN, excluding ZS3
Parent/sub pair: keeping INN1, excluding 6GF
Parent/sub pair: keeping ASG, excluding B7A
Parent/sub pair: keeping KDB, excluding KB9
LOW_DATA exclusions: ['FTD', '1ZG', '7CA', '2X1', 'EZ4']

Companies excluded: 24
Eligible for portfolio: 140
Exclusion log saved.


,ticker,reason
0,ZS3,Subsidiary of CRIN — excluded to prevent carbo...
1,6GF,Subsidiary of INN1 — excluded to prevent carbo...
2,B7A,Subsidiary of ASG — excluded to prevent carbon...
3,KB9,Subsidiary of KDB — excluded to prevent carbon...
4,FTD,No ESG data (all three pillars NaN) — excluded...
5,1ZG,No ESG data (all three pillars NaN) — excluded...
6,7CA,No ESG data (all three pillars NaN) — excluded...
7,2X1,No ESG data (all three pillars NaN) — excluded...
8,EZ4,No ESG data (all three pillars NaN) — excluded...
9,SEBA,ESG score below floor (48.3 < 50.1)


## Step 3 â€” Composite ranking score

We combine ESG quality and financial quality into one ranking score.  
Adjust the weights to reflect your portfolio mandate.

In [4]:
ESG_WEIGHT    = 0.55
SHARPE_WEIGHT = 0.30
BIO_WEIGHT    = 0.10
EU_WEIGHT     = 0.05

# Sharpe normalised 0-100
s_min = eligible['sharpe_ratio'].min()
s_max = eligible['sharpe_ratio'].max()
eligible['sharpe_score'] = ((eligible['sharpe_ratio'] - s_min) / (s_max - s_min) * 100).round(2)

# Biodiversity: invert so lower risk = higher score
if 'biodiversity_score' in eligible.columns and eligible['biodiversity_score'].notna().any():
    b_min = eligible['biodiversity_score'].min()
    b_max = eligible['biodiversity_score'].max()
    eligible['bio_score_inv'] = ((1 - (eligible['biodiversity_score'] - b_min) / (b_max - b_min + 1e-9)) * 100).round(2)
else:
    eligible['bio_score_inv'] = 50
    BIO_WEIGHT = 0.0
    ESG_WEIGHT += 0.05
    SHARPE_WEIGHT += 0.05
    print('No biodiversity data — weight redistributed.')

# EU Taxonomy eligibility normalised 0-100
eu_elig_col = next((c for c in eligible.columns if 'eligib' in c.lower()), None)
if eu_elig_col:
    eligible['eu_score'] = pd.to_numeric(eligible[eu_elig_col], errors='coerce').fillna(0)
    eu_max = eligible['eu_score'].max()
    eligible['eu_score'] = (eligible['eu_score'] / eu_max * 100).round(2) if eu_max > 0 else 50
else:
    eligible['eu_score'] = 50
    EU_WEIGHT = 0.0
    ESG_WEIGHT += 0.025
    SHARPE_WEIGHT += 0.025

# Renormalise weights to sum to 1
total_w = ESG_WEIGHT + SHARPE_WEIGHT + BIO_WEIGHT + EU_WEIGHT
ESG_WEIGHT    /= total_w
SHARPE_WEIGHT /= total_w
BIO_WEIGHT    /= total_w
EU_WEIGHT     /= total_w

eligible['composite_score'] = (
    eligible['ESG_score']     * ESG_WEIGHT +
    eligible['sharpe_score']  * SHARPE_WEIGHT +
    eligible['bio_score_inv'] * BIO_WEIGHT +
    eligible['eu_score']      * EU_WEIGHT
).round(2)

eligible = eligible.sort_values('composite_score', ascending=False).reset_index(drop=True)
eligible['rank'] = eligible.index + 1

print(f'Weights: ESG={ESG_WEIGHT:.0%}  Sharpe={SHARPE_WEIGHT:.0%}  Bio={BIO_WEIGHT:.0%}  EU={EU_WEIGHT:.0%}')
print('\nTop 25 by composite score:')
eligible[[TICKER_COL, 'ESG_score', 'sharpe_score', 'bio_score_inv', 'eu_score', 'composite_score', 'rank']].head(25)


Weights: ESG=55%  Sharpe=30%  Bio=10%  EU=5%

Top 25 by composite score:


,ticker,ESG_score,sharpe_score,bio_score_inv,eu_score,composite_score,rank
0,G7W,76.22,100.00,85.0,0.00,80.42,1
1,AVS,75.87,77.40,100.0,0.00,74.95,2
2,1SQ,75.58,72.54,70.0,0.00,70.33,3
3,1AE,62.38,93.33,70.0,0.00,69.31,4
4,ASME,66.45,71.64,100.0,0.00,68.04,5
5,9TG,73.64,65.08,70.0,0.00,67.03,6
6,SKWA,79.57,37.85,70.0,90.68,66.65,7
7,CJ2,71.42,65.88,70.0,0.00,66.04,8
8,SOAN,72.19,63.84,70.0,0.00,65.86,9
9,IHCB,86.22,34.80,70.0,0.00,64.86,10


## Step 4 — Correlation Filter

Walk down the ranked list and drop any candidate whose return correlation with an already-selected stock exceeds the threshold.  
This ensures diversification without discarding high-ESG names arbitrarily.

In [5]:
N_HOLDINGS     = 20     # number of final holdings (also used in Step 5)
CORR_THRESHOLD = 0.85   # drop a candidate if it correlates > 0.85 with any already-selected stock
CANDIDATE_POOL = 40     # consider top N ranked stocks before correlation filter

# Load price data to build return correlation matrix
price_files = sorted(glob.glob("../data/market/prices_*.csv"))
if not price_files:
    print("No price file found — skipping correlation filter, using ranked order.")
    eligible_filtered = eligible.head(N_HOLDINGS * 2).copy()
else:
    prices = pd.read_csv(price_files[-1], index_col=0, parse_dates=True)
    daily_returns = prices.pct_change().dropna(how="all")

    # Map yf_ticker back to our ticker column (if column exists)
    if "yf_ticker" in eligible.columns:
        ticker_map = eligible[[TICKER_COL, "yf_ticker"]].dropna().drop_duplicates()
        available = ticker_map[ticker_map["yf_ticker"].isin(daily_returns.columns)]
    else:
        # yf_ticker not in merge — try using ticker column directly
        ticker_map = eligible[[TICKER_COL]].copy()
        ticker_map["yf_ticker"] = ticker_map[TICKER_COL]
        available = ticker_map[ticker_map["yf_ticker"].isin(daily_returns.columns)]

    # Build correlation matrix for candidate pool
    candidates = eligible.head(CANDIDATE_POOL).copy()
    candidates = candidates.merge(available, on=TICKER_COL, how="left", suffixes=("", "_map"))
    yf_col = "yf_ticker_map" if "yf_ticker_map" in candidates.columns else "yf_ticker"
    pool_tickers = candidates[yf_col].dropna().tolist()
    ret_pool = daily_returns[[t for t in pool_tickers if t in daily_returns.columns]]
    corr_matrix = ret_pool.corr() if not ret_pool.empty else pd.DataFrame()

    # Greedy selection: walk ranked list, skip if too correlated with already selected
    selected_yf = []
    selected_bb = []

    for _, row in candidates.iterrows():
        yf_t = row.get(yf_col)
        bb_t = row[TICKER_COL]
        if pd.isna(yf_t) or corr_matrix.empty or yf_t not in corr_matrix.columns:
            selected_yf.append(yf_t)
            selected_bb.append(bb_t)
        elif not selected_yf:
            selected_yf.append(yf_t)
            selected_bb.append(bb_t)
        else:
            existing_with_data = [t for t in selected_yf if t is not None and t in corr_matrix.columns]
            if existing_with_data:
                max_corr = corr_matrix.loc[yf_t, existing_with_data].abs().max()
                if max_corr > CORR_THRESHOLD:
                    print(f"Dropped {bb_t} (max corr {max_corr:.2f} with existing selection)")
                    continue
            selected_yf.append(yf_t)
            selected_bb.append(bb_t)
        if len(selected_bb) >= N_HOLDINGS:
            break

    # Remove duplicates (same bb ticker selected twice) — shouldn't happen but guard
    seen = set()
    deduped_bb = []
    for t in selected_bb:
        if t not in seen:
            seen.add(t)
            deduped_bb.append(t)
    selected_bb = deduped_bb

    eligible_filtered = eligible[eligible[TICKER_COL].isin(selected_bb)].copy()
    print(f"After correlation filter: {len(eligible_filtered)} candidates selected from top {CANDIDATE_POOL}")

eligible_filtered[[TICKER_COL, "composite_score", "rank"]].head(25)

After correlation filter: 20 candidates selected from top 40


C:\Users\ionva\AppData\Local\Temp\ipykernel_47052\3769091267.py:12: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  daily_returns = prices.pct_change().dropna(how="all")


,ticker,composite_score,rank
0,G7W,80.42,1
1,AVS,74.95,2
2,1SQ,70.33,3
3,1AE,69.31,4
4,ASME,68.04,5
5,9TG,67.03,6
6,SKWA,66.65,7
7,CJ2,66.04,8
8,SOAN,65.86,9
9,IHCB,64.86,10


## Step 5 — Select final holdings and assign weights

We take the top N from the correlation-filtered list.  
Weights are proportional to composite score, capped at 10% per holding.

In [6]:
# N_HOLDINGS and MAX_WEIGHT
MAX_WEIGHT = 0.10         # 10% cap per stock

portfolio = eligible_filtered.head(N_HOLDINGS).copy()

# Score-proportional weights
total_score = portfolio["composite_score"].sum()
portfolio["weight_raw"] = portfolio["composite_score"] / total_score

# Cap at MAX_WEIGHT and redistribute excess
capped = portfolio["weight_raw"].clip(upper=MAX_WEIGHT)
excess = portfolio["weight_raw"].sum() - capped.sum()
uncapped_mask = portfolio["weight_raw"] < MAX_WEIGHT
if uncapped_mask.any() and capped[uncapped_mask].sum() > 0:
    capped[uncapped_mask] += excess * (capped[uncapped_mask] / capped[uncapped_mask].sum())
portfolio["weight"] = (capped / capped.sum()).round(4)  # normalise to exactly 1.0

print(f"Final portfolio: {len(portfolio)} holdings")
print(f"Weights sum to: {portfolio['weight'].sum():.4f} (should be 1.0000)")
print(f"Max weight: {portfolio['weight'].max():.1%}")
print()
show_cols = [c for c in [TICKER_COL, "idBbGlobalCompanyName", "ESG_score", "sharpe_ratio",
                          "composite_score", "nature_risk_tier", "taxonomy_eligible_pct", "weight"]
             if c in portfolio.columns]
portfolio[show_cols]

Final portfolio: 20 holdings
Weights sum to: 1.0000 (should be 1.0000)
Max weight: 6.2%



,ticker,idBbGlobalCompanyName,ESG_score,sharpe_ratio,composite_score,nature_risk_tier,taxonomy_eligible_pct,weight
0,G7W,Games Workshop Group PLC,76.22,1.123,80.42,Low,NaN,0.0618
1,AVS,ASM International NV,75.87,0.923,74.95,Low,0.00,0.0576
2,1SQ,Swissquote Group Holding SA,75.58,0.880,70.33,Low,NaN,0.0541
3,1AE,Argenx SE,62.38,1.064,69.31,Low,NaN,0.0533
4,ASME,ASML Holding NV,66.45,0.872,68.04,Low,0.00,0.0523
5,9TG,Gaztransport Et Technigaz SA,73.64,0.814,67.03,Low,0.00,0.0515
6,SKWA,SSAB AB,79.57,0.573,66.65,Low,90.68,0.0512
7,CJ2,Ringkjoebing Landbobank A/S,71.42,0.821,66.04,Low,NaN,0.0508
8,SOAN,UnipolSai Assicurazioni SpA,72.19,0.803,65.86,Low,0.00,0.0506
9,IHCB,SBM Offshore NV,86.22,0.546,64.86,Low,NaN,0.0499


## Step 6 — Portfolio summary statistics

In [7]:
weighted_esg   = (portfolio["ESG_score"] * portfolio["weight"]).sum()
weighted_sharpe = (portfolio["sharpe_ratio"] * portfolio["weight"]).sum()

# Universe averages for comparison
universe_esg   = combined["ESG_score"].mean()
universe_sharpe = combined["sharpe_ratio"].mean()

print("=== PORTFOLIO vs UNIVERSE ===")
print(f"{'Metric':<25} {'Portfolio':>12} {'Universe':>12} {'Improvement':>12}")
print("-" * 65)
print(f"{'Weighted ESG Score':<25} {weighted_esg:>12.1f} {universe_esg:>12.1f} {weighted_esg - universe_esg:>+12.1f}")
print(f"{'Weighted Sharpe Ratio':<25} {weighted_sharpe:>12.3f} {universe_sharpe:>12.3f} {weighted_sharpe - universe_sharpe:>+12.3f}")
print(f"{'Number of Holdings':<25} {len(portfolio):>12} {len(combined):>12}")

=== PORTFOLIO vs UNIVERSE ===
Metric                       Portfolio     Universe  Improvement
-----------------------------------------------------------------
Weighted ESG Score                70.8         63.0         +7.8
Weighted Sharpe Ratio            0.776        0.562       +0.215
Number of Holdings                  20          164


## Step 7 — Save portfolio


In [8]:
today = str(date.today())
out_path = f"../outputs/portfolio/final_portfolio_{today}.csv"
portfolio.to_csv(out_path, index=False)
print(f"Portfolio saved: {out_path}")

# Also save universe scores for benchmark comparison
combined.to_csv(f"../outputs/portfolio/universe_scores_{today}.csv", index=False)
print(f"Universe scores saved.")

Portfolio saved: ../outputs/portfolio/final_portfolio_2026-05-14.csv
Universe scores saved.


## âœ… Notebook complete

You now have:
- **Final 20-stock portfolio** with weights that sum to 100%
- **Exclusion log** with reasons for every rejected company
- **Portfolio vs. universe comparison** (ESG improvement, Sharpe improvement)

**Next:** Open `05_reporting.ipynb` to generate the charts and factsheet.